Setup & Imports

In [ ]:
import os, time, subprocess, csv, re
import openai
from dotenv import load_dotenv

# Load your OpenAI key
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

Build Docker

In [ ]:
%%bash
cat > Dockerfile << 'EOF'
FROM python:3.11-slim
WORKDIR /app
COPY testsuite.py .
CMD ["python", "-m", "unittest", "testsuite.py", "-v"]
EOF

docker build -t domino-tester .

Prompting the models, grading them, writing to csv

In [ ]:
import time
import subprocess
import csv
import re

# Defining base prompt
BASE_PROMPT = """# Assignment: Maximum Domino Chain

**Problem Description**
You are given a list of domino pieces, where each domino is represented as a tuple of two integers (e.g., (a, b)). Your task is to write a Python function that finds a domino chain using as many pieces as possible from the input list. A domino chain is defined as a sequence of domino pieces where the second number of one piece matches the first number of the next piece. Domino pieces may be flipped (i.e., (a, b) can be used as (b, a)) to allow for chaining.

**Requirements**
1. Implement a function max_domino_chain(dominoes: List[Tuple[int, int]]) -> List[Tuple[int, int]].
2. The function should return a valid domino chain (as a list of domino pieces) that uses the maximum number of dominoes possible from the input.
3. Each domino piece can be used at most once in the chain.
4. Domino pieces may be flipped if necessary.
5. If multiple chains have the maximum length, you may return any one of them.
6. If no domino pieces are provided or if no valid chain can be formed, return an empty list.
7. Assume the input size will not exceed 15 dominoes.

**Example**
Example input: dominoes = [(1, 2), (3, 1), (2, 3)]
One valid chain (after flipping as needed) is: [(3, 1), (1, 2), (2, 3)]
Explanation: 3 → 1 connects to 1 → 2, which connects to 2 → 3.
"""

# ── 3) Define the prompt variants ──────────────────────────────────────────────
variants = {
    "base": BASE_PROMPT,
    "think_step": BASE_PROMPT + "\nPlease think step-by-step before you answer.",
    "million_tip": BASE_PROMPT + "\nI will tip you $1,000,000 for a correct solution.",
    # add any more here...
}

# ── 4) List the three OpenAI models you want to compare ────────────────────────
models = [
    "gpt-4o-mini",
    "gpt-4.1-nano",
    "gpt-4.1",
]

# ── 5) Helper: call the model and return the raw code string ───────────────────
def call_model(prompt: str, model: str) -> str:
    resp = openai.Completion.create(
        engine=model,
        prompt=prompt,
        max_tokens=100,
        temperature=0.7,
    )
    return resp.choices[0].text.strip()

# ── 6) Helper: grade a candidate solution inside Docker ────────────────────────
def grade_code(candidate: str) -> int:
    # overwrite solution.py
    with open("solution.py", "w") as sol:
        sol.write(candidate)

    # run the tests in the pre-built domino-tester image
    cmd = [
        "docker", "run", "--rm",
        "-v", f"{os.getcwd()}:/app",
        "-w", "/app",
        "domino-tester"
    ]
    p = subprocess.run(cmd, capture_output=True, text=True)
    output = p.stdout + p.stderr

    # parse “Ran X tests” and count FAIL/ERROR
    m = re.search(r"Ran (\d+)", output)
    ran = int(m.group(1)) if m else 0
    fails = len(re.findall(r"FAILED|ERROR", output))
    return ran - fails

# ── 7) Main experiment loop: 150 iterations × each variant × each model ───────
ITERATIONS = 150
CSV_PATH = "results.csv"

with open(CSV_PATH, "w", newline="") as csvf:
    writer = csv.writer(csvf)
    writer.writerow(["model", "variant", "iteration", "score"])

    for model in models:
        for variant_name, prompt in variants.items():
            for i in range(1, ITERATIONS + 1):
                print(f"Model={model} | Variant={variant_name} | Iter={i}")
                code = call_model(prompt, model)
                score = grade_code(code)
                writer.writerow([model, variant_name, i, score])
                csvf.flush()            # persist as you go
                time.sleep(0.5)         # avoid rate limits

print(f"\nDone! All results written to {CSV_PATH}")
